# Level 4C — Returns-Based Style Analysis

**Audience:** analysts who want to infer how a fund's returns behave relative
to investable style indices.

**Prerequisites:** Level 1, labelled pandas return series, and basic portfolio
weight interpretation.

**Learning goals**

1. distinguish return behavior from disclosed holdings;
2. estimate long-only, fully invested style exposures;
3. interpret fitted returns, residuals, and R-squared;
4. track changing exposures with rolling windows;
5. recognize index-selection, missing-data, and stability risks.

**Outline:** synthetic styles → static exposure → diagnostics → missing data →
rolling exposure → holdout review → exercise.

The notebook uses synthetic monthly decimal simple returns. It requires no
credentials, private data, or network access.

## 1. Setup and model contract

Returns-based style analysis finds a passive style-index mix whose return
variation most closely follows the fund:

\[
\min_w \operatorname{Var}(r_{fund} - R_{style}w)
\quad\text{subject to}\quad
\sum_j w_j=1,\; 0\leq w_j\leq1.
\]

The estimated weights describe **return behavior over the sample**. They are
not a reconstruction of portfolio holdings.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

pd.options.display.float_format = "{:.4f}".format

from asset_management_toolkit.analytics import (
    rolling_style_exposures,
    style_exposures,
)

## 2. Create three synthetic style indices

The styles have different volatility and co-movement. Distinct return patterns
are essential: nearly identical indices cannot identify separate exposures.

In [ ]:
generator = np.random.default_rng(42)
n_months = 72
dates = pd.date_range("2020-01-31", periods=n_months, freq="ME")

market = generator.normal(0.006, 0.035, n_months)
rate_factor = generator.normal(0.002, 0.012, n_months)
style_returns = pd.DataFrame(
    {
        "Global Equity": market + generator.normal(0.0, 0.010, n_months),
        "Government Bond": (
            -0.15 * market
            + rate_factor
            + generator.normal(0.0, 0.004, n_months)
        ),
        "Cash": generator.normal(0.0015, 0.0005, n_months),
    },
    index=dates,
)

style_returns.describe().loc[["mean", "std"]].T

## 3. Create a fund with a changing style

The first 36 months are equity-heavy. The final 36 months are bond-heavy.
A small constant selection return and residual noise are added so the example
is realistic without changing the intended style definition.

In [ ]:
early_weights = pd.Series(
    {"Global Equity": 0.65, "Government Bond": 0.25, "Cash": 0.10}
)
late_weights = pd.Series(
    {"Global Equity": 0.25, "Government Bond": 0.60, "Cash": 0.15}
)

true_weights = pd.DataFrame(
    np.vstack(
        [
            np.repeat(early_weights.to_numpy()[None, :], 36, axis=0),
            np.repeat(late_weights.to_numpy()[None, :], 36, axis=0),
        ]
    ),
    index=dates,
    columns=style_returns.columns,
)
selection_return = 0.001
residual_noise = generator.normal(0.0, 0.002, n_months)
fund_returns = (
    (style_returns * true_weights).sum(axis=1)
    + selection_return
    + residual_noise
).rename("Synthetic Fund")

pd.concat(
    [
        true_weights.iloc[[0, -1]].set_axis(["early", "late"]),
        pd.Series(
            {
                "early": fund_returns.iloc[:36].mean(),
                "late": fund_returns.iloc[36:].mean(),
            },
            name="mean_fund_return",
        ),
    ],
    axis=1,
)

## 4. Estimate one full-sample style

One estimate summarizes the complete period. Because the fund changed style
halfway through, these weights should be interpreted as an average behavioral
exposure rather than a stable mandate.

In [ ]:
full_sample = style_exposures(fund_returns, style_returns)

pd.DataFrame(
    {
        "estimated_weight": full_sample.weights,
        "early_true_weight": early_weights,
        "late_true_weight": late_weights,
    }
)

## 5. Review fit and residuals

R-squared measures in-sample return variation explained by the style mix.
Residuals retain their mean: the optimizer minimizes tracking variance rather
than forcing the average selection return to zero.

In [ ]:
pd.Series(
    {
        "observations": full_sample.n_observations,
        "r_squared": full_sample.r_squared,
        "centered_residual_sum_squares": full_sample.residual_sum_squares,
        "mean_residual": full_sample.residuals.mean(),
        "residual_volatility": full_sample.residuals.std(ddof=1),
    },
    name="fit_diagnostic",
)

In [ ]:
pd.concat(
    [
        fund_returns,
        full_sample.fitted_returns,
        full_sample.residuals,
    ],
    axis=1,
).head()

## 6. Confirm that average selection return does not change style

Adding a constant to every fund return changes the residual mean but not the
tracking variance. A correct Sharpe-style objective therefore leaves the
estimated exposures unchanged.

In [ ]:
shifted = style_exposures(fund_returns + 0.01, style_returns)

pd.DataFrame(
    {
        "original": full_sample.weights,
        "fund_plus_1pct_each_month": shifted.weights,
        "difference": shifted.weights - full_sample.weights,
    }
)

## 7. Handle missing observations explicitly

The API aligns inputs by index and jointly drops rows with any missing fund or
style return. It never replaces a missing return with zero. Always review the
resulting observation count.

In [ ]:
styles_with_gap = style_returns.copy()
styles_with_gap.loc[dates[10], "Government Bond"] = np.nan
fund_with_shorter_history = fund_returns.iloc[2:]

complete_case = style_exposures(
    fund_with_shorter_history,
    styles_with_gap,
)
pd.Series(
    {
        "fund_rows": len(fund_with_shorter_history),
        "style_rows": len(styles_with_gap),
        "complete_rows_used": complete_case.n_observations,
    }
)

## 8. Estimate rolling exposures

A 24-month trailing window is long enough to estimate the three styles while
remaining responsive to the synthetic regime change. `step=3` reports one
estimate every three complete months.

In [ ]:
rolling = rolling_style_exposures(
    fund_returns,
    style_returns,
    window=24,
    step=3,
)

rolling.weights.iloc[[0, len(rolling.weights) // 2, -1]]

In [ ]:
rolling_diagnostics = pd.concat(
    [rolling.r_squared, rolling.residual_sum_squares],
    axis=1,
)
rolling_diagnostics.tail()

## 9. Compare rolling estimates with known synthetic exposures

The first reported window belongs entirely to the early regime. The last
window belongs entirely to the late regime. Intermediate windows blend both.

In [ ]:
comparison = pd.DataFrame(
    {
        "first_rolling": rolling.weights.iloc[0],
        "early_true": early_weights,
        "last_rolling": rolling.weights.iloc[-1],
        "late_true": late_weights,
    }
)
comparison

## 10. Chronological holdout review

An in-sample R-squared is not evidence that exposures persist. Fit only the
first 36 months, freeze those weights, and evaluate the next 12 months after
the synthetic style change begins.

In [ ]:
estimation = style_exposures(
    fund_returns.iloc[:36],
    style_returns.iloc[:36],
)
holdout_styles = style_returns.iloc[36:48]
holdout_fund = fund_returns.iloc[36:48]
holdout_fitted = holdout_styles @ estimation.weights
holdout_residual = holdout_fund - holdout_fitted

pd.Series(
    {
        "estimation_r_squared": estimation.r_squared,
        "holdout_residual_mean": holdout_residual.mean(),
        "holdout_residual_volatility": holdout_residual.std(ddof=1),
    }
)

## 11. Exercise — compare window lengths

Estimate rolling exposures with 12- and 36-month windows using `step=3`.
Compare:

1. how quickly equity exposure responds after month 36;
2. the variability of each estimated weight;
3. the median rolling R-squared.

Which window is more responsive, and which is more stable?

In [ ]:
# Try it here.
short_window = rolling_style_exposures(
    fund_returns,
    style_returns,
    window=12,
    step=3,
)
long_window = rolling_style_exposures(
    fund_returns,
    style_returns,
    window=36,
    step=3,
)

### Answer scaffold

In [ ]:
pd.DataFrame(
    {
        "12_month_weight_std": short_window.weights.std(),
        "36_month_weight_std": long_window.weights.std(),
        "12_month_last_weight": short_window.weights.iloc[-1],
        "36_month_last_weight": long_window.weights.iloc[-1],
    }
).join(
    pd.DataFrame(
        {
            "12_month_median_r_squared": [short_window.r_squared.median()] * 3,
            "36_month_median_r_squared": [long_window.r_squared.median()] * 3,
        },
        index=style_returns.columns,
    )
)

## Interpretation, pitfalls, and extensions

- Estimated weights describe return behavior, not holdings.
- Index selection defines the answer. Use broad, investable, economically
  distinct style indices in consistent currencies and return conventions.
- High R-squared is in-sample fit, not proof of manager skill or persistence.
- Short windows react faster but usually produce noisier exposures.
- Long windows are smoother but can conceal real style changes.
- Do not fill missing returns with zero; review complete-case sample size.
- Similar or redundant style indices can make exposures non-identifiable.
- Fund fees, timing differences, stale pricing, derivatives, leverage, and
  nonlinear exposures can appear in residuals or distort weights.

Useful extensions include confidence intervals, turnover diagnostics,
structural-break tests, and walk-forward comparison against simple style
benchmarks. Each requires its own statistical contract.